In [2]:
from pathlib import *
import os
import random
random.seed(42)

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm import tqdm
import matplotlib.pyplot as plt
import json
import shap

from IDS.data.dataset import CICDDoS2019Dataset, min_max_normalize
from IDS.model_seq import SequenceFeatureCNN

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [2]:
dir_path = 'IDS/data/Processed Data/'
seq_length = 64
dataset = CICDDoS2019Dataset(dir_path, seq_length, F.normalize, [2.0,0], balanced=True, random_seed=42)

allocating...: 14it [00:18,  1.30s/it]
collecting data...: 14it [02:38, 11.30s/it]
verifying sequence labels...: 100%|██████████| 21775/21775 [00:00<00:00, 74828.02it/s]


In [3]:
seq_model = SequenceFeatureCNN(dataset.data.shape[-2], dataset.data.shape[-1])
seq_model.load_state_dict(torch.load('IDS/saved_models/sequence_models/baseline_epc150.pt'))
seq_model.to(device)
seq_model.eval()

SequenceFeatureCNN(
  (input_layer): Conv1d(69, 69, kernel_size=(5,), stride=(1,))
  (relu): ReLU()
  (conv1d_layer): Conv1d(69, 69, kernel_size=(5,), stride=(1,))
  (pooling_layer): AvgPool1d(kernel_size=(2,), stride=(2,), padding=(0,))
  (dense_layer1): Linear(in_features=1932, out_features=128, bias=True)
  (relu_dense): ReLU()
  (dense_layer2): Linear(in_features=128, out_features=13, bias=True)
)

In [4]:
explainer = shap.DeepExplainer(seq_model, dataset.data)
shap_values = explainer.shap_values(dataset.data)

shap_values

KeyboardInterrupt: 